# Finite-size scaling — $h_c(L)$ and extrapolation
Fit BFFM $\\rho_Z$ and Rényi $S_2$ vs $h_z$ per $L$ with a symmetric tanh and an asymmetric Richards curve; $h_c$ = inflection. Then extrapolate $h_c(L)=h_c+b\\,L^{-x}$.

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
NQS = Path("../results/nqs")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

In [ ]:
def sweep(L):                                   # {hz: data} for finite (non-diverged) runs
    out = {}
    for p in sorted(NQS.glob(f"G-equiv_1_L{L}_hx0.00_hz*.json")):
        hz = float(re.search(r"hz([\d.]+)\.json", p.name).group(1))
        E = np.array([complex(x).real for x in json.load(open(p))["energy"]])
        if np.isfinite(E[-1]):
            out[hz] = json.load(open(p))
    return out

Ls   = [6, 8, 10, 12]
data = {L: sweep(L) for L in Ls}
cols = dict(zip(Ls, plt.cm.plasma(np.linspace(0, 0.85, len(Ls)))))

bffm_z = lambda d: d["order_params"]["WilsonBFFM"][-1][6] if d["order_params"]["WilsonBFFM"] else np.nan
renyi2 = lambda d: d["order_params"]["renyi2_entropy"][-1][0] if d["order_params"]["renyi2_entropy"] else np.nan
OBS = {"BFFM $\\rho_Z$": bffm_z, "Rényi $S_2$": renyi2}

def curve(L, fn):
    hz = np.array(sorted(data[L]))
    return hz, np.array([fn(data[L][h]) for h in hz])

tanh_f = lambda h, A, B, hc, w: A + B * np.tanh((h - hc) / w)
rich_f = lambda h, A, K, h0, w, nu: A + (K - A) / (1 + np.exp(-(h - h0) / w))**nu

def fit(x, y, form):
    """return (h_c, h_c_err, popt); errors from fit covariance (underestimated)."""
    if form == "tanh":
        p0 = [y.mean(), (y[-1] - y[0]) / 2, x[len(x)//2], 0.05]
        b  = ([-np.inf, -np.inf, x.min(), 1e-4], [np.inf, np.inf, x.max(), 1.0])
        popt, pcov = curve_fit(tanh_f, x, y, p0=p0, bounds=b, maxfev=40000)
        return popt[2], np.sqrt(pcov[2, 2]), popt
    p0 = [y[0], y[-1], x[len(x)//2], 0.03, 1.0]
    b  = ([-np.inf, -np.inf, x.min(), 1e-4, 1e-2], [np.inf, np.inf, x.max(), 1.0, 50])
    popt, pcov = curve_fit(rich_f, x, y, p0=p0, bounds=b, maxfev=60000)
    A, K, h0, w, nu = popt
    hc = h0 + w * np.log(nu)                      # inflection (NOT h0 when nu != 1)
    g  = np.array([0, 0, 1, np.log(nu), w / nu])
    return hc, float(np.sqrt(g @ pcov @ g)), popt

In [ ]:
# --- knob: hz window used by the fits ---
HZ_FIT = (0.25, 0.50)      # (lo, hi); Richards needs ~>=8 points to be stable

In [ ]:
HC = {}                                          # (obs, form) -> {L: (h_c, err)}
fig, axes = plt.subplots(len(OBS), len(Ls), figsize=(3.2*len(Ls), 3.0*len(OBS)), squeeze=False)
for r, (oname, fn) in enumerate(OBS.items()):
    for c, L in enumerate(Ls):
        ax = axes[r][c]; hz, y = curve(L, fn)
        m = (hz >= HZ_FIT[0]) & (hz <= HZ_FIT[1]); x, yy = hz[m], y[m]
        ax.plot(hz, y, "o", color="0.7", ms=3)
        ax.plot(x, yy, "o", color=cols[L], ms=4)
        xs = np.linspace(x.min(), x.max(), 200)
        for form, ff, ls in [("tanh", tanh_f, "--"), ("rich", rich_f, "-")]:
            try:
                hc, err, popt = fit(x, yy, form)
                ax.plot(xs, ff(xs, *popt), ls, color=cols[L], lw=1.2)
                ax.axvline(hc, color=cols[L], ls=ls, lw=0.8, alpha=0.6)
                HC.setdefault((oname, form), {})[L] = (hc, err)
            except Exception as e:
                print(f"{oname} L={L} {form}: fit failed ({e})")
        if r == 0: ax.set_title(f"L={L}")
        if c == 0: ax.set_ylabel(oname)
        ax.set_xlabel("$h_z$")
fig.tight_layout()
# fig.savefig(FIG / "fss_fits.png", dpi=300, bbox_inches="tight")

In [ ]:
print(f"h_c(L) over hz in {HZ_FIT}  (— errors from fit covariance, underestimated)\n")
for (oname, form), d in HC.items():
    row = "  ".join(f"L{L}={d[L][0]:.4f}±{d[L][1]:.4f}" for L in Ls if L in d)
    print(f"{oname:14s} [{form:4s}]  {row}")

In [ ]:
# --- knobs ---
FIX_X    = 1/0.629         # exponent: fix (1/nu_3DIsing ~ 1.59) or None to fit it (degenerate with 4 L)
H_THEORY = 0.328473        # expected 3D-Ising* critical field, h_c = 0.328473(2)

def fss(Ls_, hc, err, fix_x):
    iL = 1.0 / np.array(Ls_, float)
    s  = np.array(err) if np.all(np.isfinite(err)) and np.all(np.array(err) > 0) else None
    if fix_x is None:
        f = lambda t, hcinf, b, x: hcinf + b * t**x; p0 = [min(hc), max(hc) - min(hc), 1.5]
    else:
        f = lambda t, hcinf, b: hcinf + b * t**fix_x; p0 = [min(hc), max(hc) - min(hc)]
    popt, pcov = curve_fit(f, iL, hc, p0=p0, sigma=s, absolute_sigma=s is not None, maxfev=40000)
    return popt, np.sqrt(np.diag(pcov)), f

ocol = {"BFFM $\\rho_Z$": "tab:blue", "Rényi $S_2$": "tab:red"}   # observable -> colour
omrk = {"BFFM $\\rho_Z$": "o",        "Rényi $S_2$": "s"}         # observable -> marker

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
tt = np.linspace(0, 1/min(Ls), 200)
print(f"h_c(inf) = h_c + b·L^(-x)   [x {'fixed='+format(FIX_X,'.3f') if FIX_X else 'fitted'}]\n")
for ax, form in zip(axes, ["tanh", "rich"]):
    for oname, fn in OBS.items():
        d = HC[(oname, form)]; Lk = [L for L in Ls if L in d]
        hc = [d[L][0] for L in Lk]; er = [d[L][1] for L in Lk]
        popt, perr, f = fss(Lk, hc, er, FIX_X)
        for L in Lk:                                                    # points: fill = L colour
            ax.errorbar(1/L, d[L][0], yerr=d[L][1], fmt=omrk[oname], mfc=cols[L], mec="k",
                        mew=0.4, ms=7, ecolor="0.5", capsize=2, zorder=3)
        ax.plot(tt, f(tt, *popt), "--", color=ocol[oname], lw=1.4, label=oname)   # dashed extrap
        ax.errorbar(0, popt[0], yerr=perr[0], fmt=omrk[oname], mfc=ocol[oname], mec=ocol[oname],
                    ms=9, ecolor=ocol[oname], capsize=4, zorder=4)      # extrapolated value + error
        print(f"  {form:4s}  {oname:14s}  {popt[0]:.4f} ± {perr[0]:.4f}")
    ax.axhline(H_THEORY, color="k", ls="--", lw=1, label=f"theory {H_THEORY}")
    ax.set(xlabel="$1/L$", title=form, ylim=(0.30, 0.48))
axes[0].set_ylabel("$h_c(L)$")
axes[0].legend(fontsize=8, loc="upper left")
fig.tight_layout()
# fig.savefig(FIG / "fss_extrapolation.png", dpi=300, bbox_inches="tight")

In [ ]:
# --- debug: what does it take for h_c(L) to reach the KNOWN h_c? ---
# The paper fits 4 points by FIXING the exponent (universality) and fitting h_c+b
# (2 params, stable). So the question is not the fit but the systematic gap.
HC_MANUAL = {6: (0.443, 0.020), 8: (0.427, 0.015), 10: (0.415, 0.010), 12: (0.402, 0.008)}
H_TARGET  = 0.328473       # known h_c(inf)
X_TARGET  = 1/0.629        # known exponent 1/nu (3D-Ising)
B_MANUAL  = None           # None -> fit b; or set a number to draw a curve by hand

Lk = sorted(HC_MANUAL); iL = 1/np.array(Lk, float)
y  = np.array([HC_MANUAL[L][0] for L in Lk]); e = np.array([HC_MANUAL[L][1] for L in Lk])

gt = lambda t, b: H_TARGET + b*t**X_TARGET            # (A) target-locked: fit only b
if B_MANUAL is None:
    (bt,), cbt = curve_fit(gt, iL, y, p0=[((y-H_TARGET)/iL**X_TARGET).mean()], sigma=e, absolute_sigma=True)
    bt_e = np.sqrt(cbt[0, 0])
else:
    bt, bt_e = B_MANUAL, 0.0
gd = lambda t, hc, b: hc + b*t**X_TARGET              # (B) data-driven: fit h_c+b (x fixed)
pd, cd = curve_fit(gd, iL, y, p0=[y.min(), bt], sigma=e, absolute_sigma=True); pd_e = np.sqrt(np.diag(cd))
resid = y - gt(iL, bt)

print(f"(A) fix h_c={H_TARGET}, x={X_TARGET:.2f}; fit b  ->  b = {bt:.3f} ± {bt_e:.3f}")
print(f"(B) fix x={X_TARGET:.2f}; fit h_c,b        ->  h_c = {pd[0]:.4f} ± {pd_e[0]:.4f}   (where the data extrapolates)")
print(f"    gap (data - target) = {pd[0]-H_TARGET:+.4f}")
print("residuals of data from the target-locked curve (systematic sign => needs subleading term / cleaner estimator):")
for L, r, ee in zip(Lk, resid, e):
    print(f"    L={L:2d}: {r:+.4f}  ({r/ee:+.1f} sigma)")

fig, (ax, axr) = plt.subplots(1, 2, figsize=(11, 4.3))
tt = np.linspace(0, iL.max()*1.05, 200)
ax.errorbar(iL, y, yerr=e, fmt="o", color="0.25", ms=6, capsize=3, zorder=3, label="$h_c(L)$")
ax.plot(tt, gt(tt, bt),  "-",  color="tab:green",  lw=1.6, label=f"target-locked (b={bt:.2f})")
ax.plot(tt, gd(tt, *pd), "--", color="tab:orange", lw=1.6, label=f"data fit $\\to$ {pd[0]:.3f}")
ax.plot(0, H_TARGET, "k*", ms=14, zorder=4); ax.axhline(H_TARGET, color="k", ls=":", lw=1, label=f"target {H_TARGET}")
ax.set(xlabel="$1/L$", ylabel="$h_c(L)$", title="scaling: data vs known $h_c$",
       ylim=(min(H_TARGET, y.min())-0.015, y.max()+0.02))
ax.legend(fontsize=8)
axr.axhline(0, color="k", lw=0.8)
axr.errorbar(iL, resid, yerr=e, fmt="o", color="tab:red", ms=6, capsize=3)
axr.set(xlabel="$1/L$", ylabel="data $-$ target curve", title="residuals")
fig.tight_layout()
# fig.savefig(FIG / "fss_debug.png", dpi=300, bbox_inches="tight")